# Chapter 4: Transfer Learning for Text Classification
**Module 03 - Deep Learning for Text with PyTorch**  
*Source integrated from `chapter4.pdf`*

This notebook completes the module with transfer learning, BERT, Transformer encoders, attention mechanisms, adversarial robustness, and course wrap-up.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain transfer learning and why it helps text classification.
- Fine-tune a BERT classifier using Hugging Face Transformers.
- Build a compact Transformer encoder classifier in PyTorch.
- Add attention to an RNN-style model.
- Describe adversarial attacks and defenses for text models.


## 4.1 What Is Transfer Learning?

Transfer learning reuses knowledge from one task for a related task. In the PDF's analogy, an English teacher who starts teaching History already brings language expertise instead of starting from zero.

| Benefit | Why it matters |
|---|---|
| Saves time | You do not train a large language model from scratch. |
| Shares expertise | The model transfers general language knowledge. |
| Reduces data needs | Smaller labeled datasets can still work well. |


## 4.2 Mechanics of Transfer Learning

```text
Pre-trained model -> Freeze or reuse base layers -> Add task-specific head -> Fine-tune
```

Typical steps:

1. Load a large pre-trained model.
2. Reuse its language representation layers.
3. Add or configure a classifier head for your task.
4. Fine-tune on a smaller labeled dataset.


## 4.3 Pre-Trained Model: BERT

**BERT** stands for **Bidirectional Encoder Representations from Transformers**.

| Feature | Detail |
|---|---|
| Architecture | Multiple Transformer encoder layers. |
| Training | Pre-trained for language modeling on large text corpora. |
| Bidirectionality | Uses left and right context at the same time. |
| Text classification | A classification head maps BERT representations to labels. |


In [ ]:
# PDF snippet: implementing BERT for sentiment classification
# Install if needed: pip install transformers
import torch

try:
    from transformers import BertTokenizer, BertForSequenceClassification

    texts = [
        "I love this!",
        "This is terrible.",
        "Amazing experience!",
        "Not my cup of tea.",
    ]
    labels = [1, 0, 1, 0]

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2,
    )

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=32,
    )
    inputs["labels"] = torch.tensor(labels)

    print("BERT batch keys:", inputs.keys())
except Exception as exc:
    print("BERT setup skipped:", exc)


In [ ]:
# PDF snippet: fine-tuning BERT
if "model" in globals() and "inputs" in globals():
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.00001)

    model.train()
    for epoch in range(1):
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch: {epoch + 1}, Loss: {loss.item():.4f}")
else:
    print("BERT fine-tuning skipped because the model was not loaded.")


In [ ]:
# PDF snippet: evaluating on new text
if "model" in globals() and "tokenizer" in globals():
    model.eval()
    text = "I had an awesome day!"
    input_eval = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
    )

    with torch.no_grad():
        outputs_eval = model(**input_eval)

    predictions = torch.nn.functional.softmax(outputs_eval.logits, dim=-1)
    predicted_label = "positive" if torch.argmax(predictions) > 0 else "negative"
    print(f"Text: {text}\nSentiment: {predicted_label}")
else:
    print("BERT evaluation skipped because the model was not loaded.")


## 4.4 Transformers for Text Processing

The PDF motivates Transformers with three strengths:

- Speed through parallel sequence processing.
- Ability to connect words regardless of distance.
- More human-like responses when trained at scale.

| Component | Role |
|---|---|
| Encoder | Processes input data. |
| Decoder | Reconstructs or generates output. |
| Feed-forward network | Refines learned representations. |
| Positional encoding | Preserves word order. |
| Multi-head attention | Captures different relationships at once. |


In [ ]:
# PDF snippet: preparing data with a train-test split
import torch.nn as nn
import torch.optim as optim

sentences = [
    "I love this product",
    "This is terrible",
    "Could be better",
    "This is the best",
]
labels = [1, 0, 0, 1]

train_sentences = sentences[:3]
train_labels = labels[:3]
test_sentences = sentences[3:]
test_labels = labels[3:]

vocab = sorted({token.lower() for sentence in sentences for token in sentence.split()})
token_embeddings = {
    token: torch.randn(1, 512)
    for token in vocab
}


In [ ]:
# PDF snippet: building the transformer model
class TransformerEncoder(nn.Module):
    def __init__(self, embed_size, heads, num_layers, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_size,
            nhead=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_size, 2)

    def forward(self, x):
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.fc(x)


transformer_model = TransformerEncoder(
    embed_size=512,
    heads=8,
    num_layers=1,
    dropout=0.1,
)
optimizer = optim.Adam(transformer_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


In [ ]:
# PDF snippet: training the transformer
for epoch in range(5):
    for sentence, label in zip(train_sentences, train_labels):
        tokens = sentence.lower().split()
        data = torch.stack([token_embeddings[token] for token in tokens], dim=1)
        output = transformer_model(data)
        loss = criterion(output, torch.tensor([label]))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


In [ ]:
# PDF snippets: predicting with the transformer
def predict(sentence):
    transformer_model.eval()
    with torch.no_grad():
        tokens = sentence.lower().split()
        data = torch.stack(
            [
                token_embeddings.get(token, torch.randn(1, 512))
                for token in tokens
            ],
            dim=1,
        )
        output = transformer_model(data)
        predicted = torch.argmax(output, dim=1)
        return "Positive" if predicted.item() == 1 else "Negative"


sample_sentence = "This product can be better"
print(f"'{sample_sentence}' is {predict(sample_sentence)}")


## 4.5 Attention Mechanisms for Text Generation

Attention assigns importance to words so the model can align its interpretation with human meaning.

Example ambiguity from the PDF:

> `"The monkey ate that banana because it was too hungry"`

The word `it` needs context. Self-attention assigns significance to words within a sentence. Multi-head attention is like multiple simultaneous views of the same sentence, capturing different relationships.


In [ ]:
# PDF snippet: attention vocabulary and data setup
data = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the bird flew over the tree",
]

vocab = set(" ".join(data).split())
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}
vocab_size = len(vocab)

pairs = [sentence.split() for sentence in data]
input_data = [[word_to_ix[word] for word in sentence[:-1]] for sentence in pairs]
target_data = [word_to_ix[sentence[-1]] for sentence in pairs]

inputs = [torch.tensor(seq, dtype=torch.long) for seq in input_data]
targets = torch.tensor(target_data, dtype=torch.long)


In [ ]:
# PDF snippets: RNN with attention model, forward pass, and padding
embedding_dim = 10
hidden_dim = 16


class RNNWithAttentionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embeddings(x)
        out, _ = self.rnn(x)
        attn_weights = torch.nn.functional.softmax(
            self.attention(out).squeeze(2),
            dim=1,
        )
        context = torch.sum(attn_weights.unsqueeze(2) * out, dim=1)
        out = self.fc(context)
        return out


def pad_sequences(batch):
    max_len = max(len(seq) for seq in batch)
    return torch.stack([
        torch.cat([seq, torch.zeros(max_len - len(seq)).long()])
        for seq in batch
    ])


In [ ]:
# PDF snippets: attention training and evaluation
criterion = nn.CrossEntropyLoss()
attention_model = RNNWithAttentionModel()
optimizer = torch.optim.Adam(attention_model.parameters(), lr=0.01)

for epoch in range(300):
    attention_model.train()
    optimizer.zero_grad()
    padded_inputs = pad_sequences(inputs)
    outputs = attention_model(padded_inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

for input_seq, target in zip(input_data, target_data):
    input_test = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0)
    attention_model.eval()
    with torch.no_grad():
        attention_output = attention_model(input_test)
    attention_prediction = ix_to_word[torch.argmax(attention_output).item()]

    print(f"\nInput: {' '.join([ix_to_word[ix] for ix in input_seq])}")
    print(f"Target: {ix_to_word[target]}")
    print(f"RNN with Attention prediction: {attention_prediction}")


## 4.6 Adversarial Attacks on Text Classification Models

Adversarial attacks are calculated changes to input data that can drastically affect an AI system's decision-making.

| Attack | Core idea |
|---|---|
| Fast Gradient Sign Method (FGSM) | Uses model gradient information to make tiny deceptive changes. |
| Projected Gradient Descent (PGD) | An iterative attack that searches for an effective disturbance. |
| Carlini & Wagner (C&W) | Optimizes the loss while trying to remain hard to detect. |

Robustness matters when systems classify user comments, moderate toxic content, or produce information that users may trust.


## 4.7 Building Defenses

The PDF lists several defense strategies:

| Defense | Purpose |
|---|---|
| Model ensembling | Use multiple models so one brittle decision boundary is less decisive. |
| Robust data augmentation | Add varied training data to reduce exploitable gaps. |
| Adversarial training | Train on adversarial examples so the model anticipates attacks. |
| Robustness tooling | Use libraries such as adversarial robustness toolboxes for testing. |
| Gradient masking | Make exploitable gradient patterns harder to use, while validating that robustness is real. |
| Regularization | Encourage balanced, less overfit models. |


## Module Wrap-Up

| Chapter | Core contribution |
|---|---|
| Chapter 1 | Foundations of text preprocessing and encoding. |
| Chapter 2 | Text classification with embeddings, CNNs, RNNs, LSTMs, GRUs, and metrics. |
| Chapter 3 | Text generation with RNNs, GANs, pre-trained models, BLEU, and ROUGE. |
| Chapter 4 | Transfer learning, BERT, Transformers, attention, and robustness. |

### Key Takeaways

- Encoding choices shape what a model can learn.
- Deep learning models for text include CNNs, RNNs, GANs, Transformers, and attention-based architectures.
- Pre-trained models make advanced NLP practical with smaller task-specific datasets.
- Robustness is part of production-quality NLP, not an afterthought.

### Suggested Next Steps

- Build a text completion project.
- Build a chatbot-style generation demo.
- Train and evaluate a sentiment analysis model.
- Continue with LLM-focused PyTorch and Transformer courses.
